# পাঠ ০১ - এআই এজেন্টদের পরিচিতি

**শুরুয়াতিদের জন্য এআই এজেন্টস** কোর্সের প্রথম পাঠে আপনাকে স্বাগতম!

একটি **এআই এজেন্ট** হল একটি প্রোগ্রাম যা একটি বড় ভাষা মডেল (এলএলএম) কে তার যুক্তি ইঞ্জিন হিসেবে ব্যবহার করে এবং বাস্তব জগতে *কার্য* সম্পাদন করতে পারে — যেমন এপিআই কল করা, ডেটাবেসে প্রশ্ন করা, বা কোড চালানো — ব্যবহারকারীর পক্ষে একটি লক্ষ্য অর্জনের জন্য।

এই নোটবুকে আপনি আপনার প্রথম এজেন্ট তৈরি করবেন: একটি **ট্র্যাভেল এজেন্ট** যা ছুটির গন্তব্যস্থান সুপারিশ করে। এর মধ্যে আপনি শিখবেন কিভাবে:

১. **মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক** ব্যবহার করে Azure AI Foundry Agent Service এর সাথে সংযোগ স্থাপন করতে হয়।
২. এজেন্টকে একটি **টুল** দিতে হয় — একটি সাধারণ পাইথন ফাংশন যা এটি কল করতে পারে।
৩. এজেন্ট চালানো এবং এর প্রতিক্রিয়া পরিদর্শন করা।
৪. এজেন্টের প্রতিক্রিয়া টোকেন-বাই-টোকেন স্ট্রিম করা।


## সেটআপ

এই নোটবুক চালানোর আগে, নিশ্চিত করুন আপনার কাছে:

1. **একটি Azure AI Foundry প্রকল্প** যার মধ্যে একটি ডিপ্লয়্ড চ্যাট মডেল রয়েছে (যেমন `gpt-4o-mini`)।
2. **Azure CLI দিয়ে লগইন করা হয়েছে** — টার্মিনালে `az login` চালান।
3. **প্রয়োজনীয় পরিবেশ ভেরিয়েবল সেট করা হয়েছে:**
   - `AZURE_AI_PROJECT_ENDPOINT` — আপনার Azure AI Foundry প্রকল্পের এন্ডপয়েন্ট।
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — আপনার ডিপ্লয়্ড মডেলের নাম।

নিচের সেলটি আপনার প্রয়োজনীয় পাইথন প্যাকেজগুলি ইনস্টল করবে।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## আপনার প্রথম এজেন্ট তৈরি করা

একজন এজেন্টের দুটি জিনিস প্রয়োজন:

- **নির্দেশনা** যা তাকে বলে *সে কে* এবং *কিভাবে আচরণ করবে* (একটি সিস্টেম প্রম্পট)।
- **টুলস** — পাইটন ফাংশন যা `@tool` দিয়ে সজ্জিত থাকে এবং যেগুলি এজেন্ট তথ্য পাওয়ার বা কাজ করার জন্য কল করতে পারে।

নীচে আমরা একটি সরল টুল সংজ্ঞায়িত করেছি যা জনপ্রিয় পর্যটন গন্তব্যের একটি তালিকা প্রদান করে। ব্যবহারকারী যখন ভ্রমণের সুপারিশ চান তখন এজেন্ট এই টুলটি ব্যবহার করবে।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## স্ট্রিমিং প্রতিক্রিয়া

একটি আরও ইন্টারেক্টিভ অভিজ্ঞতার জন্য আপনি এজেন্টের প্রতিক্রিয়া **স্ট্রিম** করতে পারেন। সম্পূর্ণ উত্তরটির জন্য অপেক্ষা করার পরিবর্তে, এজেন্ট তৈরি হওয়া সাথে সাথে টেক্সটের অংশগুলো প্রদান করে। এটি বিশেষ করে চ্যাট ইন্টারফেসে উপকারী যেখানে আপনি রিয়েল টাইমে আউটপুট প্রদর্শন করতে চান।


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## সারাংশ

এই পাঠে আপনি শিখেছেন কিভাবে:

- **একটি প্রোভাইডার তৈরি করবেন** যা `AzureAIProjectAgentProvider` ব্যবহার করে Azure AI Foundry Agent Service-এ সংযুক্ত করে।
- **একটি টুল সংজ্ঞায়িত করবেন** `@tool` ডেকোরেটর ব্যবহার করে যাতে এজেন্ট আপনার পাইথন ফাংশনগুলি কল করতে পারে।
- **এজেন্ট চালাবেন** একটি ব্যবহারকারী বার্তা দিয়ে এবং এর প্রতিক্রিয়া প্রিন্ট করবেন।
- **প্রতিক্রিয়া স্ট্রিম করবেন** রিয়েল-টাইম আউটপুটের জন্য।

পরবর্তী পাঠে আমরা এজেন্টিক ফ্রেমওয়ার্কগুলি আরও গভীরভাবে অন্বেষণ করব এবং শিখব কিভাবে এজেন্টদের আরও শক্তিশালী টুল এবং বহু-ধাপ যুক্তি ক্ষমতা প্রদান করতে হয়।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকারোক্তি**:  
এই নথিটি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা যথাসাধ্য সঠিকতার চেষ্টা করি, তবুও স্বয়ংক্রিয় অনুবাদে ভুল বা ত্রুটি থাকতে পারে। মূল নথিটি তার নিজ ভাষায়ই প্রামাণিক উৎস বলে গণ্য করা উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ প্রয়োজন। এই অনুবাদের ব্যবহার থেকে সৃষ্ট কোনো ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ থাকব না।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
